# VadCLIP Baseline — Chạy Code Gốc Trên Colab

Notebook này chạy **`VadCLIP/baseline/`**, thư mục chỉ chứa code gốc upstream, tách hẳn khỏi
`VadCLIP/src/` (nơi đã trộn nhiều nhánh thí nghiệm).

Nó là runner thuần: mọi logic nằm trong `VadCLIP/baseline/src/`, không copy code vào notebook.

| Section | Nội dung | Bắt buộc |
|---|---|---|
| 1 | Mount Drive, cấu hình đường dẫn, helper | ✅ |
| 2 | Cài dependencies | ✅ |
| 3 | Kiểm tra file bắt buộc (kể cả ground truth) | ✅ |
| 4 | Copy feature sang runtime local | ✅ |
| 5 | Kiểm tra độ phủ feature | ✅ |
| 6 | Kiểm tra GPU + liệt kê sửa đổi so với upstream | nên chạy |
| 7 | **Train baseline — config repo (lr 2e-5)** | ✅ |
| 8 | Đánh giá checkpoint cuối | ✅ |
| 9 | Copy sản phẩm về Drive | ✅ |
| 10 | (Tuỳ chọn) Train config paper (lr 1e-5) + so sánh | tuỳ chọn |

> **Cần GPU.** `DistanceAdj` trong `layers.py` upstream hardcode `.to('cuda')`.
> Runtime → Change runtime type → GPU.

> **Khác biệt so với `train_shift_consistency_vadclip_colab.ipynb`:** notebook đó chạy
> `ucf_train_augment.py --lambda-consistency 0` với `--num-workers 4`. Notebook này chạy
> chính `ucf_train.py` với DataLoader mặc định (`num_workers=0`) — tức đúng luồng dữ liệu
> của upstream.

## 1. Mount Drive Và Cấu Hình

Checkpoint ghi vào **đĩa local `/content`** chứ không phải Drive: `checkpoint.pth` chứa cả
CLIP ViT-B/16 lẫn optimizer state (vài trăm MB) và được ghi ~12 lần mỗi epoch — ghi thẳng lên
Drive sẽ chậm hơn cả phần tính toán. Section 9 copy sản phẩm cuối về Drive.

Log thì ghi thẳng lên Drive theo từng dòng, nên nếu session rớt vẫn còn log để đọc.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

# ---- Đường dẫn trên Drive ----
PROJECT_ROOT  = Path('/content/drive/MyDrive/Finetune VadCLIP')
BASELINE_SRC  = PROJECT_ROOT / 'VadCLIP' / 'baseline' / 'src'
LIST_DIR      = PROJECT_ROOT / 'VadCLIP' / 'list'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
DRIVE_RESULT = PROJECT_ROOT / 'Result' / 'baseline'
LOG_DIR      = DRIVE_RESULT / 'logs'

# ---- Đường dẫn local (nhanh) ----
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT          # section 4 sẽ trỏ lại sang local
OUT_ROOT     = Path('/content/baseline_out')

# Tag cua hai lan chay. Dat o day (khong phai o cell train) de co the mo lai
# ket qua cu bang cach chi chay cell nay roi nhay thang toi section 8.1 / 10.1.
TAG_REPO  = 'repo_lr2e-5'    # config repo   (lr 2e-5)
TAG_PAPER = 'paper_lr1e-5'   # config paper  (lr 1e-5)

# ---- List tương đối + ground truth (đường dẫn tính từ baseline/src) ----
TRAIN_LIST = '../../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST  = '../../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path',         '../../list/gt_ucf.npy',
    '--gt-segment-path', '../../list/gt_segment_ucf.npy',
    '--gt-label-path',   '../../list/gt_label_ucf.npy',
]

os.chdir(BASELINE_SRC)
sys.path.insert(0, str(BASELINE_SRC))
LOG_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULT.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    '''Chạy lệnh, stream từng dòng, và ghi log lên Drive NGAY khi có dòng mới.

    Ghi tăng dần chứ không gom cuối: nếu Colab rớt giữa chừng thì log vẫn còn.
    '''
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    log_file = open(LOG_DIR / log_name, 'w', encoding='utf-8') if log_name else None
    if log_file:
        log_file.write('$ ' + ' '.join(cmd) + '\n')
        log_file.flush()
    captured = []
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            captured.append(line)
            if log_file:
                log_file.write(line)
                log_file.flush()
    finally:
        process.wait()
        if log_file:
            log_file.close()
    if log_name:
        print('Log:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return ''.join(captured)


def build_train_cmd(tag, lr='2e-5', seed=234, max_epoch=10, batch_size=64, extra=None):
    '''Dựng lệnh train baseline. tag quyết định thư mục output.

    KHÔNG có --num-workers: baseline dùng DataLoader mặc định (num_workers=0) của upstream.
    Đổi con số đó sẽ đổi thứ tự lô dữ liệu và do đó đổi kết quả.
    '''
    out = OUT_ROOT / tag
    out.mkdir(parents=True, exist_ok=True)
    return PY + [
        'ucf_train.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list',  TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--lr', lr,
        '--max-epoch', max_epoch,
        '--batch-size', batch_size,
        '--model-path',          out / 'model_ucf.pth',
        '--checkpoint-path',     out / 'checkpoint.pth',
        '--save-cur-path',       out / 'model_cur.pth',
        '--epoch-checkpoint-dir', out / 'epoch_checkpoints',
    ] + list(extra or [])


def evaluate_model(tag):
    '''Chấm điểm checkpoint cuối bằng ucf_test.py (giao thức process_split chuẩn).'''
    return run_command(PY + [
        'ucf_test.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--model-path', OUT_ROOT / tag / 'model_ucf.pth',
    ], log_name=f'evaluate_{tag}.log')


print('Baseline src :', BASELINE_SRC)
print('List dir     :', LIST_DIR)
print('Feature root :', FEATURE_ROOT)
print('Output (local):', OUT_ROOT)
print('Drive result :', DRIVE_RESULT)
print('cwd          :', Path.cwd())

## 2. Cài Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy pandas

## 3. Kiểm Tra File Bắt Buộc

Bao gồm cả ba file ground truth. Chúng bị `.gitignore` nên không có trong repo — phải upload
sẵn lên Drive.

⚠️ Nếu định sinh lại: `list/make_gt_ucf.py` lọc `if '__0.npy' not in name: continue`, nhưng
`ucf_CLIP_rgbtest.csv` toàn bộ là `__5.npy` → chạy thẳng sẽ ra **file rỗng**. Phải sửa bộ lọc
thành `__5.npy` trước.

In [ ]:
required = [
    BASELINE_SRC / 'ucf_train.py',
    BASELINE_SRC / 'ucf_test.py',
    BASELINE_SRC / 'ucf_option.py',
    BASELINE_SRC / 'model.py',
    BASELINE_SRC / 'utils' / 'dataset.py',
    BASELINE_SRC / 'utils' / 'tools.py',
    BASELINE_SRC / 'utils' / 'layers.py',
    BASELINE_SRC / 'utils' / 'ucf_detectionMAP.py',
    BASELINE_SRC / 'clip' / 'clip.py',
    BASELINE_SRC / 'clip' / 'bpe_simple_vocab_16e6.txt.gz',
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
]
gt_required = [
    LIST_DIR / 'gt_ucf.npy',
    LIST_DIR / 'gt_segment_ucf.npy',
    LIST_DIR / 'gt_label_ucf.npy',
]

missing    = [str(p) for p in required    if not p.exists()]
missing_gt = [str(p) for p in gt_required if not p.exists()]

if missing:
    print('THIẾU FILE CODE/LIST:')
    for p in missing:
        print('  ', p)
if missing_gt:
    print('THIẾU GROUND TRUTH:')
    for p in missing_gt:
        print('  ', p)

if missing or missing_gt:
    raise FileNotFoundError('Upload các file còn thiếu lên Drive rồi chạy lại cell này.')

print('Đủ toàn bộ file bắt buộc.')
print('Feature trên Drive tồn tại:', DRIVE_FEATURE_ROOT.exists())
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)
print('Archive feature           :', archive if archive else 'không có (sẽ copy theo thư mục)')

## 4. Copy Feature Sang Runtime Local

Đọc trực tiếp 16.100 file `.npy` từ Drive rất chậm. Copy sang `/content` trước.
Cell này gán lại `FEATURE_ROOT` sang đường dẫn local.

In [ ]:
import shutil
import time

subprocess.run(['df', '-h', '/content'], check=False)
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)

start = time.time()
if archive is not None:
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        print('Copying archive:', archive)
        shutil.copy2(archive, local_archive)
    else:
        print('Local archive already present:', local_archive)
    print('Extracting:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive phải chứa thư mục top-level UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-ah', '--info=progress2',
                        f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'\nDone in {time.time() - start:.1f}s')
print('FEATURE_ROOT ->', FEATURE_ROOT)

## 5. Kiểm Tra Độ Phủ Feature

Xác nhận mọi file `.npy` mà list tham chiếu đều tồn tại, để không chết giữa lúc train.

Con số phải ra: train **16.100** dòng, test **290** dòng, thiếu **0**.

In [ ]:
import csv
from collections import Counter


def check_coverage(csv_path, feature_root, preview=20):
    rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
    missing = [r for r in rows if not (feature_root / r['path']).exists()]
    print(f'{csv_path.name}: rows={len(rows)}, missing={len(missing)}')
    if missing:
        print('  Thiếu theo nhãn:', dict(Counter(r['label'] for r in missing)))
        for r in missing[:preview]:
            print('   ', r['path'])
        raise FileNotFoundError(f'{csv_path.name} tham chiếu file không tồn tại.')


for p in [LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv']:
    check_coverage(p, FEATURE_ROOT)

import numpy as np
gt = np.load(LIST_DIR / 'gt_ucf.npy')
print('gt_ucf.npy độ dài:', len(gt), '| số frame bất thường:', int(gt.sum()))

## 6. GPU + Sửa Đổi So Với Upstream

Cell này in ra **chính xác** những dòng đã sửa so với VadCLIP upstream, bằng cách quét các
marker `ADDED:` / `FIXED:` trong code. Đọc để biết mình đang chạy cái gì.

Chi tiết đầy đủ nằm ở `VadCLIP/baseline/README.md`.

In [ ]:
import torch

print('CUDA khả dụng :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Cần GPU: layers.py upstream hardcode .to("cuda"). '
                       'Runtime -> Change runtime type -> GPU.')
print('torch         :', torch.__version__)

VERBATIM = ['model.py', 'crop.py', 'utils/tools.py', 'utils/layers.py',
            'utils/ucf_detectionMAP.py', 'utils/lr_warmup.py', 'clip/*']
MODIFIED = ['ucf_train.py', 'ucf_test.py', 'ucf_option.py', 'utils/dataset.py']

print('\n' + '=' * 78)
print('VERBATIM (không sửa một ký tự):')
for f in VERBATIM:
    print('   ', f)

print('\nĐÃ SỬA — toàn bộ marker trong code:')
for f in MODIFIED:
    lines = Path(f).read_text(encoding='utf-8').splitlines()
    hits = [(i + 1, l.strip()) for i, l in enumerate(lines)
            if 'ADDED:' in l or 'FIXED:' in l]
    print(f'\n  --- {f} ---')
    if not hits:
        print('      (chỉ sửa ở phần khai báo tham số, xem README)')
    for n, l in hits:
        print(f'      L{n}: {l}')
print('=' * 78)

## 7. Train Baseline — Config Repo (lr 2e-5)

Đây là cell chính. Nó chạy `ucf_train.py` gốc với đúng mặc định của repo:

```
seed 234 · lr 2e-5 · batch 64 · 10 epoch · MultiStepLR([4,8], 0.1)
visual_length 256 · attn_window 8 · prompt 10+10 · 14 lớp
DataLoader num_workers = 0   (mặc định upstream)
```

**Chạy rất lâu** — 10 epoch, mỗi epoch ~125 step và đánh giá toàn tập test ~12 lần.
Theo dõi bằng dòng `=== end of epoch N | best AUC so far: ... ===`.

Log ghi lên Drive theo từng dòng nên rớt session vẫn đọc được:
`Result/baseline/logs/train_repo_lr2e-5.log`

In [ ]:
run_command(build_train_cmd(tag=TAG_REPO, lr='2e-5', seed=234, max_epoch=10),
            log_name=f'train_{TAG_REPO}.log')

## 8. Đánh Giá Checkpoint Cuối

`ucf_train.py` đã chấm điểm trong lúc train và giữ checkpoint tốt nhất theo AUC1.
Cell này chấm lại checkpoint cuối bằng `ucf_test.py` chạy độc lập.

Bốn con số cần đọc:

| Dòng in ra | Ý nghĩa | Paper công bố |
|---|---|---|
| `AUC1` | AUC nhánh phân loại (chỉ số chính) | **88.02** |
| `AP1`  | AP nhánh phân loại | 33.56 |
| `AUC2` | AUC nhánh đối chiếu | 85.69 |
| `average MAP` | mAP trung bình qua 5 ngưỡng IoU | 6.68 |

> Nhắc lại: `ucf_test.py` ở đây đã sửa lỗi prompt viết hoa của upstream. Bản upstream chấm
> điểm bằng bộ prompt khác với lúc train nên cho ra số không so sánh được. Xem README mục 4.

In [ ]:
evaluate_model(TAG_REPO)

### 8.1 Đường Cong Theo Epoch

`ucf_train.py` chấm điểm toàn tập test ~12 lần mỗi epoch, nên log chứa sẵn cả đường cong.
Cell này bóc số từ log ra thành bảng thay vì phải cuộn tay qua hơn 1000 dòng.

Ba thứ đọc được:

- **best AUC1** — điểm cao nhất, chính là checkpoint được lưu thành `model_ucf.pth`.
  Phải khớp với `AUC1` mà section 8 vừa in ra.
- **AUC1 tốt nhất mỗi epoch** — xem model còn đang lên hay đã bão hoà.
- **`epoch` / `step` của điểm tốt nhất** — nếu rơi vào epoch 2-3 thì model overfit sớm.

> ⚠️ `ucf_train.py` chọn checkpoint theo AUC **trên chính tập test**. Đây là chọn-model-trên-test,
> nên con số cuối cùng lạc quan hơn thực tế. Đó là hành vi của upstream và giữ nguyên để so sánh
> công bằng — nhưng khi viết báo cáo thì phải nói rõ.

In [ ]:
import re
import pandas as pd


def parse_train_log(path):
    text = Path(path).read_text(encoding='utf-8')
    pattern = re.compile(
        r'epoch:\s+(\d+)\s+\|\s+step:\s+(\d+).*?'
        r'AUC1:\s+([\d.]+)\s+AP1:\s+([\d.]+).*?'
        r'AUC2:\s+([\d.]+)\s+AP2:\s+([\d.]+).*?'
        r'average MAP:\s+([\d.]+)',
        re.S)
    rows = [dict(epoch=int(a), step=int(b),
                 AUC1=float(c) * 100, AP1=float(d) * 100,
                 AUC2=float(e) * 100, AP2=float(f) * 100, avgMAP=float(g))
            for a, b, c, d, e, f, g in pattern.findall(text)]
    return pd.DataFrame(rows)


curve = parse_train_log(LOG_DIR / f'train_{TAG_REPO}.log')
if curve.empty:
    print('Chưa có điểm đánh giá nào trong log — train chưa chạy tới step 1280?')
else:
    print('Số lần đánh giá:', len(curve))
    print()
    print('AUC1 tốt nhất mỗi epoch:')
    print(curve.groupby('epoch')[['AUC1', 'AUC2', 'avgMAP']].max().round(2).to_string())

    best = curve.loc[curve['AUC1'].idxmax()]
    print()
    print(f'Điểm tốt nhất -> epoch {int(best.epoch)}, step {int(best.step)}')
    print(f'   AUC1={best.AUC1:.2f}  AP1={best.AP1:.2f}  '
          f'AUC2={best.AUC2:.2f}  AP2={best.AP2:.2f}  avgMAP={best.avgMAP:.2f}')
    print('   (đây chính là checkpoint được lưu thành model_ucf.pth)')

    curve.to_csv(DRIVE_RESULT / f'curve_{TAG_REPO}.csv', index=False)
    print()
    print('Saved:', DRIVE_RESULT / f'curve_{TAG_REPO}.csv')

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(curve.index, curve['AUC1'], label='AUC1 (phân loại)')
        ax.plot(curve.index, curve['AUC2'], label='AUC2 (đối chiếu)')
        ax.axhline(88.02, ls='--', c='gray', lw=1, label='paper 88.02 (tham chiếu)')
        for e in curve['epoch'].unique():
            ax.axvline(curve[curve.epoch == e].index[0], c='0.9', lw=0.8, zorder=0)
        ax.set_xlabel('lần đánh giá'); ax.set_ylabel('AUC (%)')
        ax.set_title(f'VadCLIP baseline — {TAG_REPO}'); ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(DRIVE_RESULT / f'curve_{TAG_REPO}.png', dpi=120)
        plt.show()
        print('Saved:', DRIVE_RESULT / f'curve_{TAG_REPO}.png')
    except Exception as exc:
        print('Bỏ qua phần vẽ:', exc)

## 9. Copy Sản Phẩm Về Drive

Checkpoint đang nằm ở `/content` (mất khi session kết thúc). Cell này copy về Drive.

`epoch_checkpoints/` khoảng vài trăm MB mỗi file × 10 epoch — chỉ copy khi thật sự cần cho
phân tích theo epoch. Mặc định tắt.

In [ ]:
COPY_EPOCH_CHECKPOINTS = False   # bật nếu cần phân tích theo từng epoch

def sync_to_drive(tag, with_epochs=COPY_EPOCH_CHECKPOINTS):
    src = OUT_ROOT / tag
    dst = DRIVE_RESULT / tag
    dst.mkdir(parents=True, exist_ok=True)
    for name in ['model_ucf.pth', 'checkpoint.pth']:
        if (src / name).exists():
            print(f'Copying {name} ({(src / name).stat().st_size / 1e6:.0f} MB) ...', flush=True)
            shutil.copy2(src / name, dst / name)
    if with_epochs and (src / 'epoch_checkpoints').exists():
        (dst / 'epoch_checkpoints').mkdir(exist_ok=True)
        for f in sorted((src / 'epoch_checkpoints').glob('*.pth')):
            print(f'Copying {f.name} ...', flush=True)
            shutil.copy2(f, dst / 'epoch_checkpoints' / f.name)
    print('Đã copy sang:', dst)
    for f in sorted(dst.rglob('*')):
        if f.is_file():
            print(f'   {f.relative_to(dst)}  ({f.stat().st_size / 1e6:.0f} MB)')

sync_to_drive(TAG_REPO)

## 10. (Tuỳ Chọn) Config Paper — lr 1e-5

Paper mục *Implementation Details* ghi learning rate cho UCF-Crime là **1 × 10⁻⁵**, nhưng
`ucf_option.py` của repo để **2 × 10⁻⁵**. Mọi thứ còn lại khớp (AdamW, batch 64, 10 epoch,
window 8, context length 20, λ = 1e-1).

Không lần train-from-scratch nào trong `docs/` chạm được 88.02 AUC mà paper công bố. Learning
rate có thể chính là mảnh còn thiếu — nên lần chạy này có giá trị thật, không phải chạy cho vui.

Đặt `RUN_PAPER_LR = True` rồi chạy. Tốn thêm đúng một lần train 10 epoch nữa.

In [ ]:
RUN_PAPER_LR = False

if not RUN_PAPER_LR:
    print('Đặt RUN_PAPER_LR = True để chạy config paper (lr 1e-5).')
else:
    run_command(build_train_cmd(tag=TAG_PAPER, lr='1e-5', seed=234, max_epoch=10),
                log_name=f'train_{TAG_PAPER}.log')
    evaluate_model(TAG_PAPER)
    sync_to_drive(TAG_PAPER)

### 10.1 Bảng So Sánh

Đọc số từ các log đánh giá đã sinh ra và gom thành một bảng.

`Paper (công bố)` là **dòng tham chiếu**, không phải cột hơn thua — nó là checkpoint tác giả
train trong môi trường của họ, sau số lần thử mà ta không quan sát được.

In [ ]:
import re
import pandas as pd


def parse_eval_log(path):
    text = Path(path).read_text(encoding='utf-8')
    def last(pattern):
        hits = re.findall(pattern, text)
        return float(hits[-1]) if hits else float('nan')
    return {
        'AUC1':    last(r'AUC1:\s+([\d.]+)') * 100,
        'AP1':     last(r'AP1:\s+([\d.]+)') * 100,
        'AUC2':    last(r'AUC2:\s+([\d.]+)') * 100,
        'AP2':     last(r'AP2:\s*([\d.]+)') * 100,
        'avg mAP': last(r'average MAP:\s+([\d.]+)'),
    }


rows = [{'run': 'Paper (công bố)', 'AUC1': 88.02, 'AP1': 33.56,
         'AUC2': 85.69, 'AP2': 26.50, 'avg mAP': 6.68}]
for log in sorted(LOG_DIR.glob('evaluate_*.log')):
    tag = log.stem.replace('evaluate_', '')
    rows.append({'run': tag, **parse_eval_log(log)})

table = pd.DataFrame(rows).set_index('run').round(2)
display(table)
table.to_csv(DRIVE_RESULT / 'baseline_comparison.csv')
print('Saved:', DRIVE_RESULT / 'baseline_comparison.csv')

## Ghi Chú

**Vì sao không có `--num-workers`.** Baseline dùng DataLoader mặc định của upstream
(`num_workers=0`). Đó là chủ ý: `ucf_train_augment.py` trong `VadCLIP/src/` chạy với
`--num-workers 4`, và điều đó làm đổi thứ tự lô dữ liệu — bằng chứng là loss ở step 0 khác nhau
(`1.55576038` với 4 worker so với `1.56109810` với 0 worker). Thư mục baseline tồn tại để giữ
đúng luồng dữ liệu gốc, nên tham số này cố tình không được đưa ra.

**Con số này so với cái gì.** Kết quả ở đây là mốc from-scratch hợp lệ trong **môi trường của
bạn**. Muốn so một phương pháp mới, hãy so với nó — đừng so với 88.02, vì làm vậy sẽ gộp cả
phần chênh lệch do môi trường vào phần "cải thiện do phương pháp".

**Một seed là chưa đủ.** Đổi `--seed` và chạy lại vài lần trước khi kết luận bất cứ điều gì về
mức tăng giảm. Đổi seed qua `build_train_cmd(tag='repo_seed1234', seed=1234)`.